# 04.02 — Comparer entraînement, reprises et évaluations

Analyse hors ligne les récompenses, les erreurs, les sorties de piste et les modèles présents. Ne communique pas avec le jeu.

In [ ]:
from pathlib import Path
import sys
CWD=Path.cwd().resolve()
REPO=next((p for p in [CWD,*CWD.parents] if (p/'assetto_corsa_gym'/'assetto_corsa_gym'/'AssettoCorsaPlugin').is_dir()),None)
if REPO is None:
    raise FileNotFoundError('Ouvrir la racine IA_projet_Ouarti_ds_github ou un sous-dossier dans VS Code.')
PROJECT=REPO/'Code'
GYM_ROOT=REPO/'assetto_corsa_gym'/'assetto_corsa_gym'
assert PROJECT.is_dir(), 'Dossier Code introuvable'
if sys.version_info[:2] != (3,9):
    raise RuntimeError('Selectionner le kernel Python 3.9.13 dans VS Code. Actuel : '+sys.version)
for name in ['datasets','models','reports']:
    (PROJECT/name).mkdir(parents=True, exist_ok=True)
print('Depot :',REPO,'\nCode :',PROJECT,'\nPython :',sys.executable)

In [ ]:
from pathlib import Path
import json, pandas as pd, numpy as np, matplotlib.pyplot as plt
models=sorted((PROJECT/'models').glob('*.zip'))
print('Modeles:',len(models))
for model in models:print(' -',model.name,round(model.stat().st_size/1024/1024,2),'Mo')
logs=sorted((PROJECT/'reports').glob('**/ppo_steps.jsonl'))
print('Journaux RL:',len(logs))
for log in logs:
    data=[]
    with log.open(encoding='utf8') as f:
        for line in f:
            if line.strip():
                try:data.append(json.loads(line))
                except json.JSONDecodeError:pass
    if not data:continue
    df=pd.DataFrame(data)
    print('\n',log.parent.name,'transitions:',len(df))
    print('Reward moyen:',round(float(df.reward.mean()),4))
    for key in ['offtrack','collision','lap_finished','idle']:
        if key in df:print(key,int(df[key].fillna(False).sum()))
    df.reward.rolling(100,min_periods=1).mean().plot(figsize=(10,3),
        title='Recompense PPO, moyenne mobile 100 — '+log.parent.name)
    plt.xlabel('Transition');plt.ylabel('Recompense');plt.grid(True);plt.show()

In [ ]:
files=sorted((PROJECT/'reports').glob('evaluation_*.csv'))
if files:
    all_results=[]
    for path in files:
        df=pd.read_csv(path)
        df['fichier']=path.name
        all_results.append(df)
    reports=pd.concat(all_results,ignore_index=True)
    display(reports)
    display(reports.groupby('fichier').agg(
        reward_moyen=('reward','mean'),episodes=('reward','size'),
        pas_moyens=('steps','mean')).round(2))
else:
    print('Aucun rapport d evaluation: activer 04.01 apres entrainement')